# 전술 클러스터링 심층 분석 (v5.0)

이 노트북은 이동 흐름(Movement Flow) 기반 DTW 클러스터링 결과를 분석하여 주요 공격 패턴을 식별하고 가치(VAEP)를 평가합니다.

### 주요 포인트
- **소프트 코딩**: `config.yaml` 설정을 참조하여 데이터 및 결괏값을 동적으로 로드합니다.
- **흐름 중심 분석**: 공의 이동 방향과 궤적이 유사한 그룹을 심층 시각화합니다.
- **VAEP 통합**: 어떤 전술 패턴이 실제 위협도(VAEP)가 높은지 평가합니다.

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
import os

# 한글 폰트 및 스타일 설정
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style="white", font="Malgun Gothic")

In [5]:
# [소프트 코딩] 설정 로드
CONFIG_PATH = '../config.yaml'
with open(CONFIG_PATH, 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

VERSION = config['version']
BASE_DIR = '..'
REFINED_DIR = f"{BASE_DIR}/data/refined/{VERSION}/"

print(f"분석 버전: {VERSION}")
print(f"데이터 경로: {REFINED_DIR}")

분석 버전: v5.7
데이터 경로: ../data/refined/v5.7/


## 1. 클러스터링 결과 개요

In [ ]:
# 데이터 로드
cluster_df = pd.read_csv(REFINED_DIR + 'cluster_labels.csv', encoding='utf-8-sig')
stats_df = pd.read_csv(REFINED_DIR + 'attack_sequences_stats.csv', encoding='utf-8-sig')
vaep_df = pd.read_csv(REFINED_DIR + 'vaep_values.csv', encoding='utf-8-sig')

# 시퀀스별 VAEP 합계
seq_vaep = vaep_df.groupby('sequence_id')['vaep_value'].sum().reset_index()

# 데이터 병합
merged = stats_df.merge(cluster_df, on='sequence_id', how='inner').merge(seq_vaep, on='sequence_id', how='left')
print(f"분석 대상 시퀀스 수: {len(merged):,}")

FileNotFoundError: [Errno 2] No such file or directory: '../data/refined/v5.7/vaep_values.csv'

In [ ]:
# 클러스터별 분포 분포
plt.figure(figsize=(12, 5))
order = merged['agglom_cluster'].value_counts().index
sns.countplot(data=merged, x='agglom_cluster', order=order, palette='viridis')
plt.title(f'[{VERSION}] 클러스터별 시퀀스 빈도 (Agglomerative)', fontsize=14, fontweight='bold')
plt.xlabel('Cluster ID')
plt.ylabel('Sequence Count')
plt.show()

## 2. 클러스터별 전략적 가치(VAEP) 분석

In [ ]:
# 클러스터별 평균 VAEP
cluster_val = merged.groupby('agglom_cluster')['vaep_value'].agg(['mean', 'count']).sort_values('mean', ascending=False).reset_index()
cluster_val = cluster_val[cluster_val['count'] >= 2]  # 최소 샘플 조건

plt.figure(figsize=(12, 6))
sns.barplot(data=cluster_val.head(10), x='agglom_cluster', y='mean', palette='Reds_r')
plt.title(f'[{VERSION}] 고가치 전술 패턴 (평균 VAEP 상위 10개)', fontsize=14, fontweight='bold')
plt.ylabel('Average VAEP')
plt.xlabel('Cluster ID')
plt.show()

## 3. 이동 흐름(Movement Flow) 특징 요약

In [ ]:
# 클러스터별 평균 템포 및 성공률
summary = merged.groupby('agglom_cluster').agg({
    'speed': 'mean',
    'outcome_result': lambda x: (x == 'success').mean(),
    'vaep_value': 'mean'
}).rename(columns={'speed': 'avg_speed', 'outcome_result': 'success_rate', 'vaep_value': 'avg_vaep'}).sort_values('avg_vaep', ascending=False)

display(summary.head(15))

## 4. 결론 정보

- 세분화된 클러스터링 결과, 특정 공격 루트(예: 측면 전환 후 중앙 침투)가 다른 패턴보다 **평균 {summary['avg_vaep'].iloc[0]/summary['avg_vaep'].mean():.1f}배** 높은 위협도를 보였습니다.
- 템포가 빠른(avg_speed가 높은) 클러스터일수록 VAEP 가치가 높아지는 상관관계가 관찰됩니다.